# Fraud Detection MLOps
## XGBoost + MLflow + Docker + Kubernetes

---

Este notebook apresenta, linha a linha e na ordem real de construção e execução, todo o pipeline MLOps de detecção de fraudes bancárias desenvolvido para a disciplina.

**O que você vai aprender:**
- O que é MLOps e por que ele existe
- Por que usamos Docker e Kubernetes (e não só um script Python)
- Como cada arquivo do projeto se encaixa no pipeline
- Como treino e serving são isolados em containers separados
- Como o MLflow faz o papel de "memória" do pipeline

---
## 1. O que é MLOps?

**MLOps** (Machine Learning Operations) é o conjunto de práticas que une o ciclo de vida do modelo de ML com as práticas de engenharia de software profissional: versionamento, reproducibilidade, deploy automatizado, monitoramento e governança.

### O problema sem MLOps

Sem MLOps, o fluxo típico é:
1. Cientista treina modelo no notebook local
2. Salva o arquivo `.pkl` em alguma pasta
3. Manda o arquivo por e-mail para o time de infra
4. Ninguém sabe quais parâmetros foram usados
5. Semanas depois ninguém consegue reproduzir o resultado
6. Em produção, o modelo é diferente do que foi testado

### O que MLOps resolve

| Problema | Solução neste projeto |
|---|---|
| "Qual versão do modelo está em prod?" | MLflow Model Registry com alias `champion` |
| "Quais params foram usados?" | MLflow loga todos os parâmetros automaticamente |
| "Funciona no meu PC, não no servidor" | Docker garante ambiente idêntico |
| "Como escalar o serving?" | Kubernetes gerencia réplicas automaticamente |
| "Como isolar treino de serving?" | Containers e Jobs separados no K8s |
| "Os dados somem quando o pod reinicia" | PersistentVolumeClaim (PVC) persiste dados |

### Arquitetura geral do projeto

```
┌─────────────────────────────────────────────────────────────┐
│               Kubernetes Cluster (namespace: fraud-detection)    │
│                                                             │
│  ┌─────────────┐    ┌──────────────────┐                   │
│  │  data-pvc   │    │   mlflow-pvc     │                   │
│  │ CSV 1M rows │    │ SQLite + artefs  │                   │
│  └──────┬──────┘    └────────┬─────────┘                   │
│         │                   │                              │
│  ┌──────▼──────┐    ┌────────▼─────────┐                   │
│  │  train Job  │───►│  MLflow Server   │◄──┐               │
│  │  (XGBoost)  │    │  (tracking+reg.) │   │               │
│  └─────────────┘    └──────────────────┘   │               │
│                                            │               │
│                     ┌──────────────────────┘               │
│                     │                                      │
│  ┌──────────────────▼──┐   ┌──────────────────────────┐   │
│  │   fraud-serving     │   │  fraud-serving-svc       │   │
│  │   (FastAPI)         │──►│  NodePort 30080          │──►│──► cliente
│  │   /health /predict  │   └──────────────────────────┘   │
│  └─────────────────────┘                                   │
└─────────────────────────────────────────────────────────────┘
```

---
## 2. Por que Docker e Kubernetes?

### Por que Docker?

Docker resolve o problema de **reproducibilidade de ambiente**. Quando você escreve:

```dockerfile
FROM python:3.11-slim-bookworm
RUN pip install xgboost==2.1.1 mlflow==2.14.3
```

...você garante que **qualquer máquina** — seu notebook, o CI/CD, o servidor de produção — vai executar exatamente com as mesmas dependências, mesma versão do Python, mesma versão de cada biblioteca.

Neste projeto usamos **3 imagens separadas**, cada uma com responsabilidade única:

| Imagem | Responsabilidade | Dependências |
|---|---|---|
| `mlflow-server` | Servidor de tracking | só mlflow |
| `fraud-train` | Treinamento XGBoost | mlflow + xgboost + sklearn + pandas |
| `fraud-serve` | API de predição | mlflow + fastapi + uvicorn |

Essa separação é um princípio fundamental de MLOps: **o container de serving não precisa do pandas ou sklearn**, e o container de treino não precisa do FastAPI. Imagens menores = menos superfície de ataque, menos dependências a gerenciar.

### Por que Kubernetes?

Kubernetes (K8s) resolve o problema de **orquestração de containers** em produção:

| Necessidade MLOps | Recurso Kubernetes |
|---|---|
| Executar treinamento como tarefa batch, uma vez | `Job` |
| Manter API de serving sempre disponível | `Deployment` |
| Persistir dados além do ciclo de vida do container | `PersistentVolumeClaim` |
| Isolar recursos de rede por projeto | `Namespace` |
| Expor API externamente com IP fixo | `Service` (NodePort/LoadBalancer) |
| Verificar se a API está saudável automaticamente | `readinessProbe` / `livenessProbe` |
| Controlar CPU/memória por container | `resources.requests/limits` |

Em resumo: Docker empacota o código; Kubernetes o executa, escala e monitora.

---
## 3. Estrutura do Projeto

```
Projeto/
│
├── docker/
│   ├── Dockerfile.mlflow   ← container do servidor MLflow
│   ├── Dockerfile.train    ← container de treinamento
│   └── Dockerfile.serve    ← container da API de serving
│
├── k8s/                    ← manifests Kubernetes (numerados na ordem de apply)
│   ├── 00-namespace.yaml
│   ├── 01-mlflow-pvc.yaml
│   ├── 02-mlflow-deployment.yaml
│   ├── 03-mlflow-service.yaml
│   ├── 04-train-job.yaml
│   ├── 05-serve-deployment.yaml
│   ├── 06-serve-service.yaml
│   ├── 07-data-pvc.yml
│   └── 08-upload-pod.yaml
│
├── src/
│   ├── train.py            ← script de treinamento
│   └── serve.py            ← API FastAPI de serving
│
├── requirements-train.txt  ← dependências do container de treino
├── requirements-serve.txt  ← dependências do container de serving
└── data/
    └── transactions_1M.csv ← dataset de transações bancárias
```

### Ordem de execução real no pipeline

```
1. Criar namespace
2. Criar PVCs (armazenamento persistente)
3. Subir MLflow (deploy + service)
4. Fazer upload do CSV para o PVC de dados
5. Rodar Job de treinamento
6. Subir API de serving (deploy + service)
```

---
## 4. O Dataset

O arquivo `data/transactions_1M.csv` simula transações bancárias com **1 milhão de linhas**.

### Features (colunas de entrada)

| Coluna | Tipo | Descrição |
|---|---|---|
| `amount` | float | Valor da transação em R$ |
| `hour` | int | Hora do dia (0–23) |
| `dow` | int | Dia da semana (0=seg, 6=dom) |
| `channel` | int | Canal: 0=app, 1=web, 2=ATM |
| `international` | int | 1 se transação internacional |
| `new_merchant` | int | 1 se comerciante nunca visto antes |
| `acct_age_days` | int | Idade da conta em dias |
| `txn_count_24h` | int | Qtd. de transações nas últimas 24h |
| `txn_amount_24h` | float | Valor total gasto nas últimas 24h |
| `distance_km` | float | Distância da última transação (km) |
| `device_change` | int | 1 se mudou de dispositivo recentemente |
| `ip_risk` | float | Score de risco do IP (0.0–1.0) |

### Target

| Coluna | Descrição |
|---|---|
| `is_fraud` | 1 = transação fraudulenta, 0 = legítima |

### Por que esse dataset é desafiador?

Fraude bancária é um problema de **classes extremamente desbalanceadas**: tipicamente apenas 0.1% a 1% das transações são fraudes. Isso faz com que um modelo que sempre preveja "não fraude" tenha 99%+ de acurácia — mas seja completamente inútil na prática.

Por isso o projeto usa:
- **AUPRC** (Area Under Precision-Recall Curve) como métrica principal
- **`scale_pos_weight`** no XGBoost para balancear as classes
- **Seleção de threshold** para garantir Precision ≥ 0.90

In [ ]:
import pandas as pd

df = pd.read_csv('data/transactions_1M.csv')

print(f'Shape: {df.shape}')
print(f'\nDistribuicao da classe target:')
print(df['is_fraud'].value_counts())
print(f'\nTaxa de fraude: {df["is_fraud"].mean():.2%}')
print(f'\nPrimeiras linhas:')
df.head()

---
## 5. Infraestrutura Kubernetes

### 5.1 Namespace — `00-namespace.yaml`

O primeiro recurso a ser criado é o **Namespace**. Em Kubernetes, um Namespace é como uma pasta isolada dentro do cluster — todos os recursos do projeto (pods, services, pvcs) vivem dentro dele.

**Por que usar Namespace?**
- Isola o projeto de outros workloads no mesmo cluster
- Permite políticas de acesso por namespace
- Facilita a limpeza: `kubectl delete namespace fraud-detection` remove tudo de uma vez

```yaml
apiVersion: v1
kind: Namespace
metadata:
  name: fraud-detection      # todos os recursos usarão namespace: fraud-detection
```

**Comando de execução:**
```bash
kubectl apply -f k8s/00-namespace.yaml

```

### 5.2 PersistentVolumeClaims — `01-mlflow-pvc.yaml` e `07-data-pvc.yml`

Containers são **efêmeros por natureza**: quando um pod morre ou reinicia, todo o sistema de arquivos interno é destruído. Para persistir dados além do ciclo de vida do container, Kubernetes usa **PersistentVolumes**.

Um **PersistentVolumeClaim (PVC)** é o "pedido" de armazenamento que um pod faz ao cluster. O cluster aloca o disco real (PersistentVolume) automaticamente.

Este projeto usa 2 PVCs:

#### PVC do MLflow — `01-mlflow-pvc.yaml`

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mlflow-pvc
  namespace: fraud-detection
spec:
  accessModes: ["ReadWriteOnce"]   # montado em apenas 1 nó por vez (suficiente para 1 replica)
  resources:
    requests:
      storage: 5Gi                 # 5 GB para banco SQLite + artefatos (modelos, imagens)
```

Este disco vai guardar:
- `mlflow.db` — banco SQLite com todos os experimentos, runs, parâmetros e métricas
- `artifacts/` — arquivos salvos pelo MLflow (modelo XGBoost, confusion matrix, metadata.json)

#### PVC de Dados — `07-data-pvc.yml`

```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: data-pvc
  namespace: fraud-detection
spec:
  accessModes: ["ReadWriteOnce"]
  resources:
    requests:
      storage: 20Gi                # 20 GB para o CSV de 1M de linhas
```

Este disco vai guardar o `transactions_1M.csv`. O Job de treinamento monta esse disco em `/data` no container.

**Comandos de execução:**
```bash
kubectl apply -f k8s/01-mlflow-pvc.yaml
kubectl apply -f k8s/07-data-pvc.yml
kubectl -n fraud-detection get pvc   # verificar status: deve ser Bound
```

### 5.3 Container do MLflow — `docker/Dockerfile.mlflow`

Antes de fazer o deploy do MLflow no Kubernetes, precisamos entender como ele é empacotado em Docker.

```dockerfile
# Imagem base: Python 3.11 minimalista (Debian Bookworm)
# Bookworm = Debian 12, com menos CVEs que a versão anterior (Bullseye)
FROM python:3.11-slim-bookworm

# Diretório de trabalho dentro do container
# Todos os arquivos gerados (mlflow.db, artifacts/) ficam aqui
WORKDIR /mlflow

# Instala apenas o mlflow — sem pandas, sklearn, etc.
# Principio de imagem mínima: só o necessário para o servidor
RUN pip install --no-cache-dir mlflow==2.14.3

# Porta padrão do MLflow UI/API
EXPOSE 5000

# Inicia o servidor com flags explícitos
# IMPORTANTE: passamos --backend-store-uri e --default-artifact-root
# diretamente como flags CLI para garantir que o servidor use o
# PVC montado em /mlflow — e não o diretório padrão ./mlruns
CMD ["mlflow", "server",
     "--backend-store-uri", "sqlite:////mlflow/mlflow.db",
     "--default-artifact-root", "/mlflow/artifacts",
     "--host", "0.0.0.0",
     "--port", "5000"]
```

**Por que SQLite como backend?**
Para um projeto acadêmico ou de desenvolvimento, SQLite é suficiente. Em produção real, usaríamos PostgreSQL como backend e um bucket S3/GCS para artefatos.

**Build:**
```bash
docker build -f docker/Dockerfile.mlflow -t thiagoar587/mlflow-server:latest .
docker push thiagoar587/mlflow-server:latest
```

### 5.4 Deploy do MLflow — `02-mlflow-deployment.yaml` e `03-mlflow-service.yaml`

#### Deployment — `02-mlflow-deployment.yaml`

```yaml
apiVersion: apps/v1
kind: Deployment           # mantém N replicas sempre rodando
metadata:
  name: mlflow
  namespace: fraud-detection
spec:
  replicas: 1              # 1 instância do MLflow server
  selector:
    matchLabels:
      app: mlflow          # o Deployment gerencia pods com esta label
  template:
    metadata:
      labels:
        app: mlflow        # label aplicada ao pod
    spec:
      containers:
        - name: mlflow
          image: docker.io/thiagoar587/mlflow-server:latest
          ports:
            - containerPort: 5000
          volumeMounts:
            - name: mlflow-vol
              mountPath: /mlflow    # monta o PVC em /mlflow dentro do container
      volumes:
        - name: mlflow-vol
          persistentVolumeClaim:
            claimName: mlflow-pvc  # referencia o PVC criado antes
```

**Por que Deployment e não Pod direto?**
Um Pod nu não se recupera automaticamente se morrer. Um Deployment garante que sempre haverá 1 réplica rodando — se o pod cair, o K8s cria outro automaticamente.

#### Service — `03-mlflow-service.yaml`

```yaml
apiVersion: v1
kind: Service
metadata:
  name: mlflow-svc          # este é o hostname que outros pods usarão
  namespace: fraud-detection
spec:
  selector:
    app: mlflow             # roteia tráfego para pods com label app=mlflow
  ports:
    - port: 5000
      targetPort: 5000
  type: ClusterIP           # só acessível de dentro do cluster
```

**O Service é fundamental:** os outros containers (train, serve) se conectam ao MLflow usando o endereço `http://mlflow-svc:5000`. O Kubernetes resolve esse hostname para o IP do pod do MLflow via DNS interno. Sem o Service, cada vez que o pod do MLflow reiniciasse, teria um IP diferente.

**Comandos:**
```bash
kubectl apply -f k8s/02-mlflow-deployment.yaml
kubectl apply -f k8s/03-mlflow-service.yaml
kubectl -n fraud-detection get pods     # aguardar status Running
```

### 5.5 Upload do Dataset — `08-upload-pod.yaml`

O CSV existe localmente (`data/transactions_1M.csv`), mas precisa estar dentro do cluster, montado no PVC `data-pvc`. A estratégia é criar um pod temporário que monta o PVC e fica aguardando — usamos então `kubectl cp` para copiar o arquivo para dentro do pod (e consequentemente para o PVC).

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: data-uploader
  namespace: fraud-detection
spec:
  restartPolicy: Never      # pod de uso único, não reinicia
  containers:
    - name: shell
      image: busybox:1.36   # imagem mínima (~1MB) com shell básico
      command: ["sh", "-c", "sleep 36000"]  # fica vivo por 10h para o upload
      volumeMounts:
        - name: data-vol
          mountPath: /data  # PVC montado aqui
  volumes:
    - name: data-vol
      persistentVolumeClaim:
        claimName: data-pvc
```

**Por que `busybox` e não a imagem de treino?**
Para upload de arquivo não precisamos de Python nem de bibliotecas ML. `busybox` tem ~1MB vs ~1GB da imagem de treino. Princípio: use a ferramenta certa para cada tarefa.

**Comandos:**
```bash
kubectl apply -f k8s/08-upload-pod.yaml
kubectl -n fraud-detection wait --for=condition=Ready pod/data-uploader

# Copia o CSV local para dentro do PVC via o pod
kubectl -n fraud-detection cp data/transactions_1M.csv data-uploader:/data/transactions_1M.csv

# Confirma que o arquivo chegou
kubectl -n fraud-detection exec data-uploader -- ls -lh /data

# Remove o pod temporário (o arquivo continua no PVC)
kubectl -n fraud-detection delete pod data-uploader
```

---
## 6. Treinamento

### 6.1 Dependências — `requirements-train.txt`

```
mlflow==2.14.3      # tracking de experimentos e model registry
xgboost==2.1.1      # algoritmo de gradient boosting
numpy==1.26.4       # arrays numéricos (matrizes de features)
pandas==2.2.2       # leitura do CSV e manipulação de DataFrame
scikit-learn==1.5.2 # métricas (AUPRC, AUROC, confusion_matrix)
matplotlib==3.9.2   # geração do gráfico da confusion matrix
```

Todas as versões são **pinadas** para garantir reproducibilidade. Se amanhã o XGBoost lançar a versão 2.2.0 com comportamento diferente, o treinamento continua idêntico porque está fixado em `2.1.1`.

### 6.2 Dockerfile de Treino — `docker/Dockerfile.train`

```dockerfile
FROM python:3.11-slim-bookworm   # base segura e minimalista
WORKDIR /app

COPY requirements-train.txt .    # copia primeiro para aproveitar o cache do Docker
RUN pip install --no-cache-dir -r requirements-train.txt
# --no-cache-dir: não salva cache do pip, mantém a imagem menor

COPY src/ src/                   # copia os scripts Python
CMD ["python", "src/train.py"]   # comando padrão: roda o treinamento
```

**Por que copiar `requirements.txt` antes do `src/`?**
Docker usa um sistema de camadas (layers). Se você copiar os requirements primeiro e instalar as dependências, essa camada fica em cache. Nas próximas builds, se só o código mudou (mas não os requirements), o Docker pula a etapa de `pip install` — muito mais rápido.

**Build:**
```bash
docker build -f docker/Dockerfile.train -t thiagoar587/fraud-train:latest .
docker push thiagoar587/fraud-train:latest
```

### 6.3 Script de Treinamento — `src/train.py`

Este é o coração do pipeline de treinamento. Vamos analisar cada seção em detalhe.

#### Imports e Configuração via Variáveis de Ambiente

```python
import os
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, confusion_matrix
)

import xgboost as xgb
import matplotlib.pyplot as plt

# Configuração via variáveis de ambiente — valores padrão para dev local
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow-svc:5000")
EXPERIMENT   = os.environ.get("MLFLOW_EXPERIMENT_NAME", "fraud-xgb-mlops")
MODEL_NAME   = os.environ.get("MLFLOW_MODEL_NAME", "fraud-xgb")
DATA_PATH    = os.environ.get("DATA_PATH", "/data/transactions_1M.csv")
SEED         = int(os.environ.get("SEED", "42"))
TARGET_PRECISION = float(os.environ.get("TARGET_PRECISION", "0.90"))
SAMPLE_N     = int(os.environ.get("SAMPLE_N", "0"))  # 0 = usa todo o dataset
```

**Por que `os.environ.get()` e não valores hardcoded?**

Esta é uma prática essencial de MLOps/12-Factor App: **separar configuração do código**. O mesmo script `train.py` funciona:
- **Localmente** (com `MLFLOW_TRACKING_URI=http://localhost:5000`)
- **No Kubernetes** (com `MLFLOW_TRACKING_URI=http://mlflow-svc:5000`)
- **Com diferentes experimentos** (trocando só a variável de ambiente)
- **Com amostras** (definindo `SAMPLE_N=10000` para testar mais rápido)

Tudo isso sem modificar uma linha do código. Os valores padrão garantem que o script funcione mesmo sem as variáveis definidas.

#### Helper: Seleção de Threshold

```python
def pick_threshold_for_precision(y_true, y_prob, target_precision: float) -> float:
    # Calcula a curva Precision-Recall para todos os thresholds possíveis
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    
    best_thr = 0.5      # valor default se nenhum threshold atender o critério
    best_recall = -1.0  # acumulador para o maior recall encontrado
    
    # Para cada ponto da curva (precision[:-1] e recall[:-1] porque
    # precision_recall_curve retorna n+1 pontos para n thresholds)
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        # Só considera thresholds que atendem o precision mínimo
        if p >= target_precision and r > best_recall:
            best_recall = r
            best_thr = float(t)  # guarda o threshold com maior recall
    
    return best_thr
```

---

**Por que isso importa?**

Um modelo de fraude não prevê diretamente "fraude" ou "não fraude" — ele prevê uma **probabilidade** (ex.: `0.73`). Precisamos decidir: acima de qual probabilidade declaramos fraude?

- **Threshold baixo (ex.: 0.3):** detecta mais fraudes, mas bloqueia muitas transações legítimas → cliente irritado
- **Threshold alto (ex.: 0.9):** quase não bloqueia transações legítimas, mas deixa passar mais fraudes → prejuízo

Esta função encontra automaticamente o threshold que **maximiza o recall mantendo precision ≥ `target_precision`**. Em linguagem de negócio: _"Bloqueamos o máximo de fraudes possível sem errar mais de 10% nas acusações."_

---

**Por que `best_recall = -1.0` e não `0.0`?**

É um truque clássico de inicialização para **garantir que qualquer valor real vença na primeira comparação**.

Recall sempre está no intervalo `[0.0, 1.0]`. Ao iniciar com `-1.0`, qualquer recall real que apareça no loop vai satisfazer `r > best_recall` na primeira iteração — o acumulador começa "impossível de ser pior".

Se usasse `best_recall = 0.0`, um threshold com recall exatamente igual a `0.0` nunca seria aceito (precisaria ser `> 0.0`), o que poderia descartar um threshold válido na borda da curva.

---

**Como a função otimiza o threshold?**

A lógica é um **filtro + maximização** que percorre todos os pontos da curva Precision-Recall:

```
Para cada (precision p, recall r, threshold t) na curva:

    ┌─ Filtro ──────────────────────────────────────────┐
    │  p >= target_precision?                           │
    │  Só considera o ponto se atende a precisão mínima │
    └───────────────────────────────────────────────────┘
              ↓ passou o filtro?
    ┌─ Maximização ─────────────────────────────────────┐
    │  r > best_recall?                                 │
    │  Esse ponto tem recall maior que o melhor atual?  │
    │  Se sim, salva esse threshold como novo candidato │
    └───────────────────────────────────────────────────┘
```

**Visualizando na curva Precision-Recall:**

```
Precision
  1.0 │  ●
      │    ●
      │      ●  ← zona válida (p >= target_precision)
target│- - - ●──●──●
      │              ●──●
  0.0 └─────────────────────────→ Recall
                              ↑
                    queremos chegar aqui
                    (maior recall dentro da zona válida)
```

A função percorre todos os pontos da zona válida (acima da linha tracejada) e retorna o threshold do ponto mais à **direita** — ou seja, aquele com maior recall que ainda garante a precisão exigida.

**Por que `precision[:-1]` e `recall[:-1]`?**

`precision_recall_curve` do scikit-learn retorna `n+1` pontos para `n` thresholds. O último ponto é artificialmente adicionado com `precision=1.0` e `recall=0.0` (sem threshold correspondente). O `[:-1]` descarta esse ponto extra para que o `zip` com `thresholds` funcione corretamente — se não fizesse isso, o índice ficaria desalinhado.

#### Helper: Carregamento e Validação do Dataset

```python
def load_dataset(csv_path: str) -> pd.DataFrame:
    # Verifica se o arquivo existe antes de tentar carregar
    # Mensagem de erro clara indica o problema (PVC não montado, path errado, etc.)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"CSV não encontrado em {csv_path}. "
            f"Monte o arquivo no container/pod e defina DATA_PATH, se necessário."
        )

    df = pd.read_csv(csv_path)

    # Valida que todas as colunas necessárias existem no CSV
    # Detecta erros de schema antes do treinamento começar
    required = {
        "amount", "hour", "dow", "channel", "international", "new_merchant",
        "acct_age_days", "txn_count_24h", "txn_amount_24h", "distance_km",
        "device_change", "ip_risk", "is_fraud"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV está sem colunas obrigatórias: {sorted(missing)}")

    # Suporte a amostragem: permite treinar com 10k linhas para testes rápidos
    if SAMPLE_N and SAMPLE_N > 0 and SAMPLE_N < len(df):
        df = df.sample(n=SAMPLE_N, random_state=SEED)  # reproducível pelo SEED

    return df
```

**Boa prática:** Validar o schema dos dados na entrada é um princípio de **Data Validation** em MLOps. Erros de schema descobertos depois de 2 horas de treinamento são muito mais custosos que erros descobertos nos primeiros segundos.

#### Função Principal: `main()`

**Preparação dos dados:**

```python
def main():
    # Conecta ao servidor MLflow (rodando no cluster ou localmente)
    mlflow.set_tracking_uri(TRACKING_URI)
    # Cria ou reutiliza o experimento com este nome
    mlflow.set_experiment(EXPERIMENT)

    df = load_dataset(DATA_PATH)

    # Separa features (X) do target (y)
    y = df["is_fraud"].values
    X = df.drop(columns=["is_fraud"])

    # Split estratificado: mantém proporção de fraudes em treino e teste
    # stratify=y é CRÍTICO em datasets desbalanceados — sem ele, o split
    # poderia colocar quase todas as fraudes no conjunto de treino
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # scale_pos_weight = neg/pos diz ao XGBoost quanto valorizar erros
    # em amostras positivas (fraudes) vs negativas
    # Ex: se há 1000 transações normais para cada fraude,
    # scale_pos_weight = 1000 — o modelo trata cada fraude
    # como se fossem 1000 exemplos normais
    pos = max(1, int(ytr.sum()))
    neg = max(1, int((ytr == 0).sum()))
    scale_pos_weight = neg / pos
```

**Parâmetros do XGBoost:**

```python
    params = {
        "objective": "binary:logistic",  # problema binário → output é probabilidade [0,1]
        "eval_metric": "aucpr",           # otimiza AUPRC durante o treino
        "max_depth": int(os.environ.get("MAX_DEPTH", "6")),         # profundidade máxima das árvores
        "eta": float(os.environ.get("ETA", "0.08")),                # learning rate
        "subsample": float(os.environ.get("SUBSAMPLE", "0.9")),     # % de linhas por árvore (regularização)
        "colsample_bytree": float(os.environ.get("COLSAMPLE", "0.9")), # % de features por árvore
        "min_child_weight": float(os.environ.get("MIN_CHILD_WEIGHT", "1.0")),
        "lambda": float(os.environ.get("LAMBDA", "1.0")),           # L2 regularization
        "alpha": float(os.environ.get("ALPHA", "0.0")),             # L1 regularization
        "scale_pos_weight": float(scale_pos_weight),
        "seed": SEED,
    }
    # Todos os hiperparâmetros podem ser sobrescritos por env var
    # → permite rodar experimentos diferentes sem modificar código
```

#### MLflow Run: Logging Completo

```python
    with mlflow.start_run() as run:
        # ── Parâmetros ──────────────────────────────────────────────────
        # Loga TUDO que influencia o resultado do treinamento
        # mlflow.log_param salva no banco SQLite para consulta futura
        mlflow.log_param("data_path", DATA_PATH)
        mlflow.log_param("rows_used", len(df))
        mlflow.log_param("target_precision", TARGET_PRECISION)
        mlflow.log_param("sample_n", SAMPLE_N)
        for k, v in params.items():
            mlflow.log_param(k, v)

        # ── Treinamento ──────────────────────────────────────────────────
        # DMatrix é o formato nativo do XGBoost: mais eficiente que DataFrame
        dtr = xgb.DMatrix(Xtr, label=ytr)
        dte = xgb.DMatrix(Xte, label=yte)

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=int(os.environ.get("NUM_BOOST_ROUND", "300")),
            # 300 árvores → cada uma corrige os erros da anterior
        )

        # ── Avaliação ────────────────────────────────────────────────────
        yprob = booster.predict(dte)   # probabilidades [0,1] para cada transação

        # AUPRC: métrica ideal para datasets desbalanceados
        # Mede a área sob a curva Precision vs Recall
        auprc = float(average_precision_score(yte, yprob))

        # AUROC: complementar — mede discriminação geral do modelo
        auroc = float(roc_auc_score(yte, yprob))

        # Encontra o threshold ótimo para precision >= TARGET_PRECISION
        thr = pick_threshold_for_precision(yte, yprob, TARGET_PRECISION)
        yhat = (yprob >= thr).astype(int)  # converte probabilidade → classe binária

        cm = confusion_matrix(yte, yhat)
        tn, fp, fn, tp = cm.ravel()
        # precision = TP / (TP + FP): de tudo que acusamos de fraude, quantos são fraude
        precision = tp / max(1, (tp + fp))
        # recall = TP / (TP + FN): de todas as fraudes reais, quantas detectamos
        recall    = tp / max(1, (tp + fn))

        # ── Métricas no MLflow ──────────────────────────────────────────
        mlflow.log_metric("auprc", auprc)
        mlflow.log_metric("auroc", auroc)
        mlflow.log_metric("precision_at_thr", float(precision))
        mlflow.log_metric("recall_at_thr",    float(recall))
        mlflow.log_metric("threshold",        float(thr))

        # ── Artefatos ────────────────────────────────────────────────────
        os.makedirs("artifacts", exist_ok=True)

        # Salva confusion matrix como imagem e loga no MLflow
        cm_path = "artifacts/confusion_matrix.png"
        plot_cm(cm, cm_path)
        mlflow.log_artifact(cm_path)   # arquivo fica disponível na UI do MLflow

        # Metadata JSON: documenta o modelo (features, tamanho, balanço de classes)
        meta = {
            "features": list(X.columns),
            "rows_used": int(len(df)),
            "class_balance_train": {"pos": int(pos), "neg": int(neg)},
        }
        with open("artifacts/metadata.json", "w") as f:
            json.dump(meta, f, indent=2)
        mlflow.log_artifact("artifacts/metadata.json")

        # ── Salvar e Registrar o Modelo ──────────────────────────────────
        # log_model salva o booster XGBoost no MLflow com o flavor nativo
        # artifact_path="model" → dentro do run, o modelo fica em runs:/id/model
        mlflow.xgboost.log_model(booster, artifact_path="model")

        client = MlflowClient()
        model_uri = f"runs:/{run.info.run_id}/model"

        # Registra no Model Registry com um nome estável "fraud-xgb"
        # Toda vez que rodar novamente, cria uma nova versão numerada (v1, v2, ...)
        mv = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

        # Define o alias "champion" para esta versão
        # API moderna do MLflow 2.9+: substitui o antigo "stage Production"
        # O alias "champion" indica: "este é o modelo campeão, use em produção"
        # A API de serving vai buscar: models:/fraud-xgb@champion
        client.set_registered_model_alias(
            name=MODEL_NAME,
            alias="champion",
            version=mv.version,
        )

        print("run_id:", run.info.run_id)
        print("registered:", MODEL_NAME, "version:", mv.version, "alias: champion")
```

**Por que `with mlflow.start_run()`?**
O bloco `with` garante que o run é encerrado corretamente mesmo se ocorrer um erro. Isso evita runs "fantasma" travados no estado RUNNING no banco MLflow.

### 6.4 Kubernetes Job — `04-train-job.yaml`

```yaml
apiVersion: batch/v1
kind: Job                    # Job = tarefa batch que roda uma vez e termina
metadata:
  name: fraud-train-job
  namespace: fraud-detection
spec:
  backoffLimit: 0            # não tenta novamente se falhar (fail-fast)
                             # em MLOps queremos saber de falhas imediatamente
  template:
    spec:
      restartPolicy: Never   # consistente com backoffLimit: 0
      containers:
        - name: train
          image: docker.io/thiagoar587/fraud-train:latest
          env:
            # Passa o endereço do MLflow via env var
            # mlflow-svc é resolvido pelo DNS interno do Kubernetes
            - name: MLFLOW_TRACKING_URI
              value: "http://mlflow-svc:5000"
            - name: MLFLOW_EXPERIMENT_NAME
              value: "fraud-xgb-mlops"
            - name: MLFLOW_MODEL_NAME
              value: "fraud-xgb"
            - name: DATA_PATH
              value: "/data/transactions_1M.csv"
            - name: TARGET_PRECISION
              value: "0.90"
            - name: NUM_BOOST_ROUND
              value: "300"
            - name: SEED
              value: "42"
            - name: SAMPLE_N
              value: "0"     # 0 = usa todo o dataset de 1M linhas

          # Limites de recursos: evita que o Job consuma todos os recursos do nó
          resources:
            requests:
              cpu: "1"       # garante 1 CPU disponível para o Job
              memory: "2Gi"  # reserva 2GB de RAM
            limits:
              cpu: "2"       # pode usar até 2 CPUs se disponível
              memory: "4Gi" # limite máximo: 4GB (XGBoost + 1M linhas)

          volumeMounts:
            - name: data-vol
              mountPath: /data    # o CSV fica visível em /data no container
              readOnly: true      # segurança: treino não modifica os dados
      volumes:
        - name: data-vol
          persistentVolumeClaim:
            claimName: data-pvc  # referencia o PVC com o CSV
```

**Por que Job e não Deployment para treinamento?**

| | Job | Deployment |
|---|---|---|
| Propósito | Tarefa que termina | Serviço contínuo |
| Quando termina | Quando o processo sai com código 0 | Nunca (reinicia sempre) |
| Uso ideal | Treinamento, migração de banco, cron | API, servidor web |

**Executar:**
```bash
kubectl apply -f k8s/04-train-job.yaml
kubectl -n fraud-detection logs -f job/fraud-train-job   # acompanhar em tempo real
```

---
## 7. Serving (API de Predição)

### 7.1 Dependências — `requirements-serve.txt`

```
mlflow==2.14.3      # para carregar o modelo do Registry
fastapi==0.112.0    # framework web moderno e de alta performance
uvicorn==0.30.6     # servidor ASGI que executa o FastAPI
numpy==1.26.4       # para converter os dados de entrada em array
pydantic==2.7.4     # validação dos dados de entrada da API
```

**Repare o que NÃO está aqui:** `pandas`, `xgboost`, `scikit-learn`, `matplotlib`. O container de serving é **muito mais leve** que o de treino. Em produção, imagens menores significam pull mais rápido, menos CVEs, startup mais veloz.

### 7.2 Dockerfile de Serving — `docker/Dockerfile.serve`

```dockerfile
FROM python:3.11-slim-bookworm
WORKDIR /app

COPY requirements-serve.txt .
RUN pip install --no-cache-dir -r requirements-serve.txt

COPY src/ src/
EXPOSE 8000

# Inicia o servidor ASGI uvicorn
# src.serve:app = módulo src.serve, objeto app (a instância FastAPI)
# --host 0.0.0.0 = aceita conexões de fora do container (necessário para K8s)
# --port 8000 = porta padrão
CMD ["uvicorn", "src.serve:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Build:**
```bash
docker build -f docker/Dockerfile.serve -t thiagoar587/fraud-serve:latest .
docker push thiagoar587/fraud-serve:latest
```

### 7.3 API de Serving — `src/serve.py`

```python
import os
import mlflow
import mlflow.pyfunc
import numpy as np
from contextlib import asynccontextmanager
from fastapi import FastAPI
from pydantic import BaseModel

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow-svc:5000")
MODEL_URI    = os.environ.get("MODEL_URI", "models:/fraud-xgb@champion")
# models:/fraud-xgb@champion = modelo "fraud-xgb" com alias "champion"
# Este alias é definido pelo train.py após cada treinamento bem-sucedido
# Para trocar para um novo modelo em produção: basta rodar novo treino
# (sem redeployar a API)

model = None  # variável global — preenchida no startup

# ── Lifespan: controla startup e shutdown da aplicação ────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    global model
    # Tudo ANTES do yield: executado no startup
    mlflow.set_tracking_uri(TRACKING_URI)
    model = mlflow.pyfunc.load_model(MODEL_URI)
    # mlflow.pyfunc.load_model carrega o modelo do Registry de forma agnóstica
    # Funciona com XGBoost, scikit-learn, PyTorch, etc. — mesma interface
    yield
    # Tudo APÓS o yield: executado no shutdown (limpeza de recursos)

# Passa o lifespan para o FastAPI
app = FastAPI(
    title="Fraud Detection API (XGBoost + MLflow)",
    lifespan=lifespan
)
```

**Por que `lifespan` em vez de carregar o modelo no nível do módulo?**

```python
# ANTES (código problemático):
model = mlflow.pyfunc.load_model(MODEL_URI)  # executa no import!
app = FastAPI(...)                           # se falhou acima, app nunca sobe
```

Com o modelo no nível do módulo, se o MLflow não estiver disponível no momento do `import`, o uvicorn trava **antes de subir o servidor**. O pod entra em `CrashLoopBackOff` e o Kubernetes não consegue nem verificar o health check.

Com `lifespan`, o FastAPI sobe primeiro. Se o carregamento do modelo falhar, o erro é logado de forma limpa e o pod pode ser inspecionado com `kubectl logs`.

#### Endpoints da API

```python
# ── Schema de entrada ─────────────────────────────────────────────────────
class PredictRequest(BaseModel):
    x: list[list[float]]  # batch de transações: [[feat1, feat2, ...], [...]]
    # Pydantic valida automaticamente: se enviar string onde espera float,
    # retorna HTTP 422 com mensagem clara de erro — sem código de validação manual

# ── GET /health ──────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_uri": MODEL_URI,
        "model_loaded": model is not None  # indica se o modelo foi carregado
    }
# Usado pelo Kubernetes readinessProbe e livenessProbe
# Se retornar HTTP 200, K8s considera o pod saudável e envia tráfego

# ── POST /predict ────────────────────────────────────────────────────────
@app.post("/predict")
def predict(req: PredictRequest):
    # Converte lista de listas para numpy array 2D
    # shape: (n_transacoes, 12_features)
    X = np.array(req.x, dtype=float)

    # model.predict via mlflow.pyfunc aceita numpy array
    # retorna array de probabilidades: [0.03, 0.97, 0.12, ...]
    preds = model.predict(X)

    # .tolist() converte numpy array para lista Python (serializável em JSON)
    return {"predictions": preds.tolist()}
```

**Exemplo de uso:**
```bash
# 12 features na ordem: amount, hour, dow, channel, international,
# new_merchant, acct_age_days, txn_count_24h, txn_amount_24h,
# distance_km, device_change, ip_risk
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"x": [[120.0, 14, 2, 1, 0, 0, 400, 2, 220.0, 3.2, 0, 0.15]]}'

# Resposta esperada:
# {"predictions": [0.023]}  → 2.3% de probabilidade de fraude
```

### 7.4 Deployment e Service do Serving — `05-serve-deployment.yaml` e `06-serve-service.yaml`

#### Deployment — `05-serve-deployment.yaml`

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fraud-serving
  namespace: fraud-detection
spec:
  replicas: 1
  selector:
    matchLabels:
      app: fraud-serving
  template:
    metadata:
      labels:
        app: fraud-serving
    spec:
      containers:
        - name: api
          image: docker.io/thiagoar587/fraud-serve:latest
          ports:
            - containerPort: 8000
          env:
            - name: MLFLOW_TRACKING_URI
              value: "http://mlflow-svc:5000"
            - name: MODEL_URI
              value: "models:/fraud-xgb@champion"  # alias atual do modelo em prod

          resources:
            requests:
              cpu: "250m"    # 0.25 CPU garantido — suficiente para serving
              memory: "512Mi"
            limits:
              cpu: "500m"    # pode chegar a 0.5 CPU em picos
              memory: "1Gi"  # modelo XGBoost em memória + overhead FastAPI

          # readinessProbe: K8s só envia tráfego quando o pod estiver pronto
          # Aguarda o modelo carregar antes de marcar como Ready
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10   # espera 10s antes da primeira verificação
            periodSeconds: 10         # verifica a cada 10s

          # livenessProbe: reinicia o pod se ficar travado
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 20   # dá tempo suficiente para o modelo carregar
            periodSeconds: 20
```

#### Service — `06-serve-service.yaml`

```yaml
apiVersion: v1
kind: Service
metadata:
  name: fraud-serving-svc
  namespace: fraud-detection
spec:
  selector:
    app: fraud-serving
  ports:
    - port: 8000
      targetPort: 8000
      nodePort: 30080      # porta fixa no IP do nó do cluster
  type: NodePort            # permite acesso externo ao cluster
                            # acesse via: http://<IP-DO-NO>:30080
```

**Diferença entre ClusterIP e NodePort:**
- `ClusterIP` (MLflow): só acessível de dentro do cluster → segurança, não expõe para fora
- `NodePort` (API): expõe na porta 30080 do nó → clientes externos conseguem chamar a API

**Comandos:**
```bash
kubectl apply -f k8s/05-serve-deployment.yaml
kubectl apply -f k8s/06-serve-service.yaml
kubectl -n fraud-detection get pods    # aguardar status Running e Ready 1/1
kubectl -n fraud-detection get svc
```

---
## 8. Pipeline Completo de Execução

Aqui está a sequência completa do zero ao modelo em produção:

### Passo 1 — Build e Push das Imagens Docker
```bash
# Construir as 3 imagens
docker build -f docker/Dockerfile.mlflow -t thiagoar587/mlflow-server:latest .
docker build -f docker/Dockerfile.train  -t thiagoar587/fraud-train:latest .
docker build -f docker/Dockerfile.serve  -t thiagoar587/fraud-serve:latest .

# Enviar para o registry (Docker Hub)
docker push thiagoar587/mlflow-server:latest
docker push thiagoar587/fraud-train:latest
docker push thiagoar587/fraud-serve:latest
```

### Passo 2 — Criar Namespace
```bash
kubectl apply -f k8s/00-namespace.yaml
```

### Passo 3 — Criar Armazenamento Persistente
```bash
kubectl apply -f k8s/01-mlflow-pvc.yaml
kubectl apply -f k8s/07-data-pvc.yaml
kubectl -n fraud-detection get pvc   # verificar: STATUS = Bound
```

### Passo 4 — Subir MLflow no Cluster
```bash
kubectl apply -f k8s/02-mlflow-deployment.yaml
kubectl apply -f k8s/03-mlflow-service.yaml
kubectl -n fraud-detection rollout status deployment/mlflow   # aguardar deploy
```

### Passo 5 — Fazer Upload do CSV para o PVC
```bash
kubectl apply -f k8s/08-upload-pod.yaml
kubectl -n fraud-detection wait --for=condition=Ready pod/data-uploader --timeout=60s
kubectl -n fraud-detection cp data/transactions_1M.csv data-uploader:/data/transactions_1M.csv
kubectl -n fraud-detection exec data-uploader -- ls -lh /data  # confirmar upload
kubectl -n fraud-detection delete pod data-uploader
```

### Passo 6 — Treinar o Modelo
```bash
kubectl apply -f k8s/04-train-job.yaml
kubectl -n fraud-detection logs -f job/fraud-train-job   # acompanhar
# Aguardar: "registered: fraud-xgb version: 1 alias: champion"
```

### Passo 7 — Subir API de Serving
```bash
kubectl apply -f k8s/05-serve-deployment.yaml
kubectl apply -f k8s/06-serve-service.yaml
kubectl -n fraud-detection rollout status deployment/fraud-serving
```

### Passo 8 — Validar
```bash
# Ver UI do MLflow
kubectl -n fraud-detection port-forward svc/mlflow-svc 5000:5000
# Abrir http://localhost:5000

# Testar API
kubectl -n fraud-detection port-forward svc/fraud-serving-svc 8000:8000
curl http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"x": [[120.0, 14, 2, 1, 0, 0, 400, 2, 220.0, 3.2, 0, 0.15]]}'
```

---
## 9. Validação e Testes

### 9.1 Verificar sintaxe dos scripts

In [ ]:
import py_compile, sys

for script in ['src/train.py', 'src/serve.py']:
    try:
        py_compile.compile(script, doraise=True)
        print(f'  OK  {script}')
    except py_compile.PyCompileError as e:
        print(f'  ERRO  {script}: {e}')

### 9.2 Entendendo as métricas

Após o treinamento, o MLflow registra 5 métricas:

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

# Simulação didática das métricas
# Em um modelo bem treinado, os valores esperados seriam aproximadamente:
metricas_esperadas = {
    'auprc':            '~0.85  (Area Under Precision-Recall Curve)\n'
                        '        → Principal métrica para fraude.\n'
                        '          Um classificador aleatório teria AUPRC ≈ taxa_fraude (ex: 0.01)\n'
                        '          Quanto mais próximo de 1.0, melhor.',

    'auroc':            '~0.97  (Area Under ROC Curve)\n'
                        '        → Mede discriminação geral.\n'
                        '          0.5 = aleatório, 1.0 = perfeito.',

    'threshold':        '~0.35  (threshold selecionado)\n'
                        '        → Acima deste valor, a transação é classificada como fraude.',

    'precision_at_thr': '~0.90  (precisão no threshold)\n'
                        '        → 90% das transações bloqueadas são realmente fraudes.',

    'recall_at_thr':    '~0.72  (recall no threshold)\n'
                        '        → 72% de todas as fraudes reais foram detectadas.',
}

for metrica, descricao in metricas_esperadas.items():
    print(f'{metrica:20s}: {descricao}\n')

### 9.3 Testando a API localmente (após port-forward)

Após executar `kubectl -n fraud-detection port-forward svc/fraud-serving-svc 8000:8000` em um terminal separado:

In [ ]:
# Execute este bloco com o port-forward ativo
import requests

BASE_URL = "http://localhost:8000"

# Teste de health
response = requests.get(f"{BASE_URL}/health")
print("Health:", response.json())

# Teste de predição
# Features na ordem: amount, hour, dow, channel, international,
#                    new_merchant, acct_age_days, txn_count_24h,
#                    txn_amount_24h, distance_km, device_change, ip_risk
transacoes = {
    "x": [
        # Transação normal: valor baixo, horário comercial, conta antiga
        [50.0,  10, 1, 0, 0, 0, 800, 1,  50.0,  0.5, 0, 0.05],
        # Transação suspeita: valor alto, madrugada, internacional, dispositivo novo, IP de risco
        [3500.0, 3, 6, 1, 1, 1,  30, 8, 4200.0, 2800.0, 1, 0.92],
    ]
}

response = requests.post(f"{BASE_URL}/predict", json=transacoes)
result = response.json()

print("\nPredições (probabilidade de fraude):")
for i, prob in enumerate(result["predictions"]):
    tipo = "SUSPEITA" if prob > 0.5 else "normal"
    print(f"  Transação {i+1}: {prob:.4f} ({tipo})")

---
## 10. Conceitos MLOps Demonstrados

Este projeto cobre os principais pilares de MLOps:

### 10.1 Reproducibilidade
- Versões pinadas em `requirements-*.txt`
- `SEED` controlado via env var
- Parâmetros completos logados no MLflow
- Imagens Docker versionadas no registry

### 10.2 Rastreabilidade (Experiment Tracking)
- Cada `mlflow.start_run()` cria um `run_id` único
- Todos os parâmetros, métricas e artefatos são associados ao run
- Possível comparar múltiplos runs na UI do MLflow
- `confusion_matrix.png` e `metadata.json` salvos como artefatos

### 10.3 Governança de Modelos (Model Registry)
- Modelo registrado com nome estável `fraud-xgb`
- Versão incrementada a cada treino (v1, v2, v3...)
- Alias `champion` indica qual versão está em produção
- Troca de modelo em produção = apenas redirecionar o alias (sem redeploy da API)

### 10.4 Separação de Responsabilidades
- 3 containers com responsabilidades únicas
- Treino e serving completamente desacoplados
- Dados em PVC separado dos containers

### 10.5 Operações de Produção
- `readinessProbe` e `livenessProbe` para health checks automáticos
- `resources.requests/limits` para estabilidade do cluster
- `restartPolicy: Never` no Job para fail-fast
- Service com DNS interno (`mlflow-svc`) para discovery de serviços

### 10.6 Configuração Externalizada
- Todo comportamento configurável via variáveis de ambiente
- Mesmo código roda localmente e em produção
- Hiperparâmetros do XGBoost controláveis sem alterar código

---

## Resumo Final

```
CÓDIGO            DOCKER              KUBERNETES
─────────         ──────────────      ─────────────────────────────
train.py    →  fraud-train:latest  →  Job (roda uma vez, termina)
serve.py    →  fraud-serve:latest  →  Deployment (roda sempre)
                mlflow:latest      →  Deployment (persiste com PVC)
                                   →  Services (DNS + exposição)
                                   →  PVCs (dados que sobrevivem restarts)
```

O resultado é um pipeline MLOps completo onde:
1. **Qualquer pessoa** pode reproduzir o treinamento com os mesmos resultados
2. **Todo experimento** fica registrado e comparável
3. **Promover um modelo** para produção é uma operação segura e rastreável
4. **A API** se mantém disponível e auto-recuperável
5. **A infraestrutura** é declarativa e versionada em código (GitOps)